<div align="center">

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/apriori3d/ico/blob/next/src/examples/ico_runtime_basics.ipynb)

**Introduction to ICO Framework Runtime**

</div>


# Runtime Monitoring with ICO Tools

This notebook demonstrates ICO's runtime monitoring capabilities using **Rich** integration tools:

- 📊 **Progress Tracking** — Visual progress bars for long-running operations
- 🖥️ **Rich Console Output** — Formatted printing with colors and styling  
- 🔄 **Runtime Lifecycle** — Deep dive into runtime states with `describe()`
- ⚙️ **Event System** — Commands, events, and message flow architecture

We'll show how to:
1. **Monitor progress** in distributed pipelines
2. **Capture and format output** from agents 
3. **Visualize runtime states** during execution
4. **Understand ICO's event-driven architecture**

Perfect for debugging, monitoring, and understanding ICO runtime behavior! 🎯

## 📚 Table of Contents

1. [⏳ Progress Tracking with RichProgressTool](#-progress-tracking-with-richprogresstool)
   - Setup Runtime Infrastructure
   - Runtime Tree Representation  
   - State Management & Commands
   - Execution with Visual Progress Bars

2. [🖥️ Rich Console Output with RichPrinterTool](#️-rich-console-output-with-richprintertool)
   - Formatted Output from Distributed Agents
   - Runtime Wrapper Integration
   - Event-Driven Message Flow

3. [📋 Summary & Key Concepts](#-summary--key-concepts)
4. [🚀 What's Next?](#-whats-next)

In [ ]:
# Install dependencies if running in Google Colab
try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("🚀 Running in Google Colab - installing dependencies...")

    # Install Apriori ICO framework from GitHub
    %pip install --upgrade -q git+https://github.com/apriori3d/ico.git@next

    print("✅ Dependencies installed successfully!")
    print("🎯 Ready to run ICO linear regression example!")
else:
    print("📱 Running locally - assuming dependencies are already installed")

## ⏳ Progress Tracking with RichProgressTool

Let's start with progress monitoring - essential for long-running ML pipelines. We'll create a slow data processing pipeline and track its progress visually.

In [ ]:
# Imports for progress tracking example
import time
from collections.abc import Iterator

from apriori.ico.core.operator import operator
from apriori.ico.core.runtime.progress import IcoProgress
from apriori.ico.core.sink import sink
from apriori.ico.core.source import source

# Create a slow data processing pipeline to demonstrate progress tracking
TOTAL_ITEMS = 8


@source()
def slow_data_source() -> Iterator[int]:
    """Simulate slow data source (e.g., file reading, API calls)."""
    for i in range(TOTAL_ITEMS):
        time.sleep(0.5)  # Simulate I/O delay
        yield i


@operator()
def heavy_computation(x: int) -> int:
    """Simulate heavy computation (e.g., model inference, data processing)."""
    time.sleep(0.3)  # Simulate computation time
    return x * x


@sink()
def save_result(x: int) -> None:
    """Simulate saving results (e.g., database write, file output)."""
    time.sleep(0.2)  # Simulate write delay
    pass


# 💡 Add progress tracking node to the computation flow
progress_tracker = IcoProgress[int](total=TOTAL_ITEMS, name="Data Processing")

# Create the computation flow: source → progress → processing → save_result
flow = slow_data_source | (progress_tracker | heavy_computation).stream() | save_result
flow.describe()

# 💡 IcoProgress is a node in computation flow that tracks progress of items passing through it.

────────────────────────────────────────── Flow plan: IcoChain | IcoSink ──────────────────────────────────────────

 Flow                           Signature                      Name            
 📚IcoSource(slow_data_source)  () → Iterator[int]                             
 ╭── for each in 🎞️ IcoStream()  Iterator[int] → Iterator[int]                  
 │   ⏳IcoProgress()            int → int                      Data Processing 
 │   heavy_computation          int → int                                      
 ╰─▸ yield                      Iterator[int]                                  
 🏁IcoSink(save_result)         Iterator[int] → ()                            

### 🛠️ Setup Runtime

`IcoProgress` utilizes runtime infrastructure and sends events to the `RichProgressTool` for collection and display.
In order to make it work, we need to:
- Setup `IcoRuntime`: a separate infrastructure that can be used for events and command flows and node state management
- Connect `RichProgressTool` to runtime 



In [ ]:
# Imports for Rich progress visualization
from apriori.ico.core.runtime.runtime import IcoRuntime
from apriori.ico.tools.progress.rich_progress_tool import RichProgressTool

# Create Rich progress instance and console
# rich_progress = Progress()
# Create progress tool
progress_tool = RichProgressTool()

# Create runtime: it takes the computation flow and provide infrastructure for runtime behavior
runtime = IcoRuntime(flow, tools=[progress_tool])

## 📊 Runtime Representation
- Runtime is represented as a tree of `IcoRuntimeNode`
- The root node is `IcoRuntime`
- All nodes with runtime behavior like `IcoProgress` are discovered from flow definition and added to runtime tree automatically

In [ ]:
# 💡 `IcoProgress` node was discovered from `flow` and added to the tree.
runtime.describe()

─────────── Runtime tree: <class 'apriori.ico.core.runtime.runtime.IcoRuntime'>(name=None, state=Idle) ────────────

 Tree                         State    Name            
 🚙IcoRuntime()               <Idle>                   
 ├──⏳IcoProgress()           <Idle>   Data Processing 
 └──🏃IcoToolBox()            <Idle>                   
     └──⏳RichProgressTool()  <Idle> 

## ⚙️ Runtime State Management
- All runtime nodes have `BaseStateModel`, which describes states and transitions
- Transitions occur as a result of an `IcoRuntimeCommand`
- Commands propagate through the runtime tree from top to bottom
- `IcoRuntime` and all `IcoRuntimeNodes` need to be in **Ready** state in order to function properly

In [4]:
# Activate the runtime: `RichProgressTool` will automatically find `IcoProgress` node
# and create a progress task for it.

runtime.activate()

Output()

In [5]:
# All nodes receive Activate command and ready for execution
runtime.describe()

─────────── Runtime tree: <class 'apriori.ico.core.runtime.runtime.IcoRuntime'>(name=None, state=Ready) ───────────

 Tree                         State     Name            
 🚙IcoRuntime()               <Ready>                   
 ├──⏳IcoProgress()           <Ready>   Data Processing 
 └──🏃IcoToolBox()            <Ready>                   
     └──⏳RichProgressTool()  <Ready> 

In [ ]:
# Run the computation flow: as items are processed, the progress will be updated in real-time.
# Updates will be visible on the progress bar above.
runtime.run()

In [7]:
# After execution, the runtime can be deactivated to clean up resources and reset state.
# The progress bar will be removed from the console.
runtime.deactivate()

In [8]:
# All nodes are in Idle state after deactivation
runtime.describe()

─────────── Runtime tree: <class 'apriori.ico.core.runtime.runtime.IcoRuntime'>(name=None, state=Idle) ────────────

 Tree                         State    Name            
 🚙IcoRuntime()               <Idle>                   
 ├──⏳IcoProgress()           <Idle>   Data Processing 
 └──🏃IcoToolBox()            <Idle>                   
     └──⏳RichProgressTool()  <Idle> 

In [9]:
# Typical usage of runtime:
runtime.activate().run().deactivate()

Output()

### Events technical details
- `IcoProgress` sent ProgressEvent via runtime API, then it bubbles via runtime tree and forwarded to `RichProgressTool` by `IcoRuntime`.
- The infrastructure for command and events propagation is fully supported across ICO framework and can be utulized in distributed scenarios with multiprocessing MPAgent and other components!


## 🖥️ Rich Console Output with RichPrinterTool

Next, let's explore formatted console output. This is especially useful for debugging distributed pipelines where different agents need to report their status.

The message from node is represented as an Event and seamlessly supported in distributed scenarios.


In [ ]:
from apriori.ico.core.runtime.printer import IcoPrinter
from apriori.ico.core.runtime.runtime_wrapper import wrap_runtime

printer = IcoPrinter()


@source()
def numbers() -> Iterator[int]:
    yield from range(5)


@operator()
def square(x: int) -> int:
    return x * x


# In order to use shared printer instance across many operators,
# we need to connect printer to the runtime.
# To do so, we can use `wrap_runtime` decorator to wrap the compute operator into Runtime node
# and inject the printer instance.


@wrap_runtime(printer)
@operator()
def print_input(x: int) -> int:
    printer(f"Input: {x}")
    return x


@wrap_runtime(printer)
@sink()
def print_result(x: int) -> None:
    printer(f"Result: {x}")


flow = numbers | (print_input | square).stream() | print_result
flow.describe()

# The printer node is not present in the computation flow explicitly, as it's wrapped inside the operator,
# but it can be seen in runtime tree.

────────────────────────────────────────── Flow plan: IcoChain | IcoSink ──────────────────────────────────────────

 Flow                           Signature                      Name 
 📚IcoSource(numbers)           () → Iterator[int]                  
 ╭── for each in 🎞️ IcoStream()  Iterator[int] → Iterator[int]       
 │   print_input                int → int                           
 │   square                     int → int                           
 ╰─▸ yield                      Iterator[int]                       
 🏁IcoSink(print_result)        Iterator[int] → ()                 

### 🛠️ Setup runtime with `RichPrinterTool`

In [11]:
# IcoRuntimeWrapper holds the IcoPrinter instance

from apriori.ico.tools.printer.rich_printer_tool import RichPrinterTool

# Create runtime with RichPrinterTool to visualize the print statements in Rich console
runtime = IcoRuntime(flow, tools=[RichPrinterTool()])
runtime.describe()

─────────── Runtime tree: <class 'apriori.ico.core.runtime.runtime.IcoRuntime'>(name=None, state=Idle) ────────────

 Tree                        State    Name 
 🚙IcoRuntime()              <Idle>        
 ├──🏃IcoRuntimeWrapper()    <Idle>        
 │   └──🖨️ IcoPrinter()       <Idle>        
 └──🏃IcoToolBox()           <Idle>        
     └──🖨️ RichPrinterTool()  <Idle> 

In [ ]:
# Run the flow and observe the print statements
runtime.activate().run().deactivate()

Input: 0

Result: 0

Input: 1

Result: 1

Input: 2

Result: 4

Input: 3

Result: 9

Input: 4

Result: 16

## 📋 Summary & Key Concepts

### Core Concepts Learned

1. **Runtime Infrastructure**
   - `IcoRuntime` provides event-driven infrastructure for monitoring and control
   - Runtime tree automatically discovers nodes with runtime behavior
   - State management with commands (Activate/Deactivate) and transitions

2. **Progress Tracking** 
   - `IcoProgress` nodes emit events during data processing
   - `RichProgressTool` captures events and displays visual progress bars
   - Perfect for monitoring long-running ML pipelines and batch processing

3. **Formatted Output**
   - `IcoPrinter` nodes emit message events for debugging and logging
   - `RichPrinterTool` provides colored, formatted console output
   - `wrap_runtime` decorator integrates printer instances into operators

4. **Event-Driven Architecture**
   - Commands propagate top-down through runtime tree
   - Events bubble up and are forwarded to appropriate tools
   - Fully distributed-ready with multiprocessing support

### Key Benefits

- **Visual Monitoring**: Real-time progress bars and formatted output
- **Distributed-Ready**: Works seamlessly with multiprocessing agents
- **Flexible Integration**: Tools can be combined for comprehensive monitoring
- **Event-Driven**: Clean separation of computation and monitoring concerns

## 🚀 What's Next?

### 🔄 **Multiprocessing**
- [Multi-processing example](src/examples/ico_mp_basic.py): Basic example of distributed computational flows
- [Parallel Multi-processing Pool example](src/examples/ico_mp_basic_pool.py): Distributed compute flows with parallel worker pools

### 🧠 **Machine Learning**
- [Linear Regression](src/examples/ml/ico_linear_regression.ipynb): ICO-based ML pipeline development
- [CIFAR-10 Classification with validation](src/examples/ml/cv/cifar/ico_cifar_complete_flow.ipynb): Complete CV pipeline replacing PyTorch DataLoader
- [CIFAR-10 Classification with worker pools](src/examples/ml/cv/cifar/ico_cifar_complete_flow_mp.py): Complete CV pipeline with parallel data processing workers
